# 03 — Extended Voltage-Grid: RF vs GRU vs CNN-GRU

## Motivation

Notebook 02 ran RF / LSTM / CNN-LSTM on the extended Tier-1 dataset.  
This notebook is the **GRU counterpart**: same data, same CV protocol, same RF baseline —  
only the recurrent cell changes (GRU vs LSTM) so the two can be compared directly.

GRU uses 3 gates instead of LSTM's 4 (no separate cell state), giving fewer params per hidden unit.  
Hidden size is raised to 47 (GRU) vs 40 (LSTM) to roughly match total parameter counts:

| Model | Hidden | Params |
|---|---|---|
| LSTM | 40 | ~14.6k |
| **GRU** | **47** | **~14.9k** |
| CNN-LSTM | 40 | ~16.3k |
| **CNN-GRU** | **47** | **~16.4k** |

## Target

`y = capacity_ahr / c_nominal_ah` — fractional SOH (= Ct / C₀). EOL at 0.8.

## Models

| Model | Input | Notes |
|---|---|---|
| **RF** | 11 charge-curve scalar descriptors | computed from (X, mask) via `vg_scalar_features` |
| **GRU** | (n_grid, 3) voltage-grid tensor | `VGGRUReg(n_features=3, hidden=47, dropout=0.35)` |
| **CNN-GRU** | (n_grid, 3) voltage-grid tensor | `VGCNNGRU(n_features=3, cnn_ch=8, hidden=47, dropout=0.35)` |

## Validation

- **GroupKFold-8** (headline): whole-cell folds, ~3 cells held out per fold across all 23 cells.
- **Leave-One-Dataset-Out (LODO)**: hold out entire NASA set (test on NASA, train on CALCE) and vice versa.

## Experiments

1. RF baseline — GroupKFold-8
2. GRU — GroupKFold-8
3. CNN-GRU — GroupKFold-8
4. GroupKFold-8 comparison table
5. LODO — all three models
6. LODO comparison table

## Setup

In [ ]:
import sys
from pathlib import Path
from importlib import reload

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

# Find project root (cross-platform, works from any directory in the project)
def find_project_root(start_path=None):
    """Find the project root by looking for marker files."""
    if start_path is None:
        start_path = Path.cwd()
    else:
        start_path = Path(start_path)
    
    for marker in ['.git', 'pyproject.toml', 'CLAUDE.md', '.claude']:
        for p in [start_path] + list(start_path.parents):
            if (p / marker).exists():
                return p
    
    raise RuntimeError(f"Could not find project root (looked for .git, pyproject.toml, etc.)")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import voltage_grid as vg
import vg_extended as vgx
import vg_models

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device selection: MPS (Apple Silicon) → CUDA → CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

# Paths
RESULTS = ROOT / "results" / "extended_tier1"
RESULTS.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "data" / "processed" / "vg_extended_tier1.npz"

# Training hyperparameters (match nb02 / nb09)
EPOCHS     = 300
PATIENCE   = 50
BATCH_SIZE = 64
LR         = 1e-3

print(f"Cache:   {CACHE}")
print(f"Results: {RESULTS}")

## Load extended dataset

In [ ]:
X, mask, y, groups, cidx, ds_groups = vgx.load_npz_ext(CACHE)

print(f"X:         {X.shape}  float32")
print(f"mask:      {mask.shape}  bool")
print(f"y:         {y.shape}  [{y.min():.3f}, {y.max():.3f}]")
print(f"groups:    {len(set(groups))} unique cells")
print(f"ds_groups: {dict(zip(*np.unique(ds_groups, return_counts=True)))}")

In [ ]:
# Map battery IDs to study groups (protocol-based)
study_group_map = {
    # NASA
    "B0005": "NASA_controlled",
    "B0006": "NASA_controlled",
    "B0007": "NASA_controlled",
    "B0018": "NASA_controlled",
    "RW1": "NASA_randomized",
    "RW9": "NASA_randomized",
    "RW13": "NASA_randomized",
    "RW14": "NASA_randomized",
    "RW15": "NASA_randomized",
    "RW16": "NASA_randomized",
    "RW17": "NASA_randomized",
    "RW19": "NASA_randomized",
    "RW20": "NASA_randomized",
    # CALCE CS2 Type 1 (0.5C)
    "CS2_8": "CALCE_CS2_Type1_0.5C",
    "CS2_21": "CALCE_CS2_Type1_0.5C",
    "CS2_33": "CALCE_CS2_Type1_0.5C",
    "CS2_34": "CALCE_CS2_Type1_0.5C",
    # CALCE CS2 Type 2 (1C)
    "CS2_35": "CALCE_CS2_Type2_1C",
    "CS2_36": "CALCE_CS2_Type2_1C",
    "CS2_37": "CALCE_CS2_Type2_1C",
    "CS2_38": "CALCE_CS2_Type2_1C",
    # CALCE CS2 Type 3 (variable discharge)
    "CS2_3": "CALCE_CS2_Type3_variable",
    "CS2_9": "CALCE_CS2_Type3_variable",
    # CALCE CX2 Type 1 (0.5C)
    "CX2_16": "CALCE_CX2_Type1_0.5C",
    "CX2_31": "CALCE_CX2_Type1_0.5C",
    "CX2_33": "CALCE_CX2_Type1_0.5C",
    "CX2_35": "CALCE_CX2_Type1_0.5C",
    # CALCE CX2 Type 2 (0.5C)
    "CX2_34": "CALCE_CX2_Type2_0.5C",
    "CX2_36": "CALCE_CX2_Type2_0.5C",
    "CX2_37": "CALCE_CX2_Type2_0.5C",
    "CX2_38": "CALCE_CX2_Type2_0.5C",
    # CALCE CX2 Type 3 (3C discharge)
    "CX2_8": "CALCE_CX2_Type3_3C",
}

# Create study_groups array aligned with groups (battery_id)
study_groups = np.array([study_group_map.get(bid, "unknown") for bid in groups])

print(f"Study groups defined: {sorted(set(study_groups))}")
print(f"Samples per study group:")
for sg in sorted(set(study_groups)):
    count = (study_groups == sg).sum()
    print(f"  {sg:30s}: {count:5d} samples")

## Coverage & sanity diagnostics

In [ ]:
# Per-cell mean coverage
for ds in np.unique(ds_groups):
    sel = ds_groups == ds
    cov = mask[sel].mean(axis=1).mean()
    print(f"  {ds:10s}  {sel.sum():5d} samples   mean coverage = {cov:.3f}")

coverage_per_sample = mask.mean(axis=1)
assert coverage_per_sample.min() > 0,     "Some samples have zero coverage!"
assert y.min() >= 0.3,                    f"y.min()={y.min():.3f} unexpectedly low"
assert y.max() <= 1.5,                    f"y.max()={y.max():.3f} unexpectedly high"

# Note: skipping strict leading-block check since some samples may have sparse coverage
# The mask semantics are: True = valid grid point, False = invalid/padded
# Gaps are acceptable for cells with variable coverage across the charge protocol

print("\nAll sanity checks passed.")

In [ ]:
# Coverage scatter + sample C-rate curves
vgrid = np.linspace(vg.V_HI, vg.V_LO, vg.N_GRID)  # descending

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

ds_color = {"nasa": "steelblue", "calce": "darkorange"}
for ds in np.unique(ds_groups):
    sel = ds_groups == ds
    cov = mask[sel].mean(axis=1)
    ax1.scatter(np.arange(sel.sum()), cov, s=3, alpha=0.4,
                label=ds, color=ds_color.get(ds, "gray"))
ax1.set_xlabel("Sample index (within group)")
ax1.set_ylabel("Coverage (fraction of grid)")
ax1.set_title("Per-sample grid coverage")
ax1.legend()

rng = np.random.default_rng(0)
sample_ids = rng.choice(len(X), size=12, replace=False)
for si in sample_ids:
    m = mask[si]
    ax2.plot(vgrid[m], X[si, m, 0], lw=0.8, alpha=0.7)
ax2.set_xlabel("Grid voltage (V, descending \u2192 lower = earlier in charge)")
ax2.set_ylabel("|C-rate| (normalized)")
ax2.set_title("12 random cycles: C-rate vs voltage grid")

fig.suptitle("Extended Tier-1 dataset \u2014 coverage diagnostics", fontsize=12)
fig.savefig(RESULTS / "03_gru_coverage_diag.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS / '03_gru_coverage_diag.png'}")

## RF baseline (charge-curve scalar features)

Same RF as in notebook 02 — included here so this notebook is self-contained  
and the comparison table uses results from the same run.

| Feature | Description |
|---|---|
| `coverage` | fraction of 128-pt grid covered |
| `v_start` | voltage at CC onset (lowest covered grid V) |
| `cc_dt` | CC phase duration (s) = t_elapsed at V_HI |
| `cc_slope` | avg voltage rise rate V/s = (V_HI − v_start) / cc_dt |
| `crate_mean/max/start` | C-rate statistics over valid grid positions |
| `temp_mean/max` | temperature statistics (°C) |
| `t_total` | alias of cc_dt |
| `r_proxy` | charge-side ohmic proxy: k×ΔV_step / crate_start |

In [ ]:
feat, feat_names = vgx.vg_scalar_features(X, mask)
print(f"feat shape: {feat.shape}")
print(f"features:   {feat_names}")
print()
display(pd.DataFrame(feat, columns=feat_names).describe().round(4))

In [ ]:
print("RF — GroupKFold-8 (by study group)")
res_rf_gkf = vgx.run_rf_grouped_cv(
    feat, y, groups, cidx,
    GroupKFold(n_splits=8),
    cv_groups=study_groups,
)
agg_rf_gkf = vg.aggregate(res_rf_gkf)
print(f"\n  MAE  {agg_rf_gkf['mae']:.4f} ± {agg_rf_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_rf_gkf['rmse']:.4f} ± {agg_rf_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_rf_gkf['r2']:.4f} ± {agg_rf_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_rf_gkf['skill']:.4f}")
print(f"  Spearman  {agg_rf_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_rf_gkf,
    "RF \u2014 GroupKFold-8 per-fold scatter (SOH)",
    save_path=RESULTS / "03_gru_rf_gkf_per_fold.png",
)

## GRU — `VGGRUReg`

Bidirectional 1-layer GRU → masked mean+max pool (4H) → Dropout → Linear.  
Architecture: `VGGRUReg(n_features=3, hidden=47, dropout=0.35)` (~14.9k params).  
GRU has 3 gates (reset, update, new) vs LSTM's 4 (input, forget, cell, output) — no cell state.

In [ ]:
reload(vg_models)
import vg_models  # noqa: F811

demo_gru = vg_models.VGGRUReg(n_features=3, hidden=47, dropout=0.35).to(DEVICE)
n_params_gru = sum(p.numel() for p in demo_gru.parameters())
print(f"VGGRUReg params: {n_params_gru:,}")

x4 = torch.tensor(X[:4]).to(DEVICE)
m4 = torch.tensor(mask[:4]).to(DEVICE)
with torch.no_grad():
    out4 = demo_gru(x4, m4)
print(f"Forward output shape: {out4.shape}  (expect (4,))")
del demo_gru, x4, m4, out4

In [ ]:
make_gru = lambda: vg_models.VGGRUReg(n_features=3, hidden=47, dropout=0.35)

print("GRU — GroupKFold-8 (by study group)")
res_gru_gkf = vgx.run_grouped_cv(
    make_gru, X, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=8),
    cv_groups=study_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler="plateau",
)
agg_gru_gkf = vg.aggregate(res_gru_gkf)
print(f"\n  MAE  {agg_gru_gkf['mae']:.4f} ± {agg_gru_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_gru_gkf['rmse']:.4f} ± {agg_gru_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_gru_gkf['r2']:.4f} ± {agg_gru_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_gru_gkf['skill']:.4f}")
print(f"  Spearman  {agg_gru_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_gru_gkf,
    "GRU \u2014 GroupKFold-8 per-fold scatter (SOH)",
    save_path=RESULTS / "03_gru_gkf_per_fold.png",
)
vgx.per_fold_loss_curves_ext(
    res_gru_gkf,
    "GRU \u2014 GroupKFold-8 train/val loss (L1)",
    save_path=RESULTS / "03_gru_gkf_loss_curves.png",
)

## CNN-GRU — `VGCNNGRU`

Conv1d front-end (kernel 5, GroupNorm, same-padding) → BiGRU → masked pool → Dropout → Linear.  
Architecture: `VGCNNGRU(n_features=3, cnn_ch=8, hidden=47, dropout=0.35)` (~16.4k params).

In [ ]:
reload(vg_models)
import vg_models  # noqa: F811

demo_cnngru = vg_models.VGCNNGRU(n_features=3, cnn_ch=8, hidden=47, dropout=0.35).to(DEVICE)
n_params_cnngru = sum(p.numel() for p in demo_cnngru.parameters())
print(f"VGCNNGRU params: {n_params_cnngru:,}")

x4 = torch.tensor(X[:4]).to(DEVICE)
m4 = torch.tensor(mask[:4]).to(DEVICE)
with torch.no_grad():
    out4 = demo_cnngru(x4, m4)
print(f"Forward output shape: {out4.shape}  (expect (4,))")
del demo_cnngru, x4, m4, out4

In [ ]:
make_cnngru = lambda: vg_models.VGCNNGRU(n_features=3, cnn_ch=8, hidden=47, dropout=0.35)

print("CNN-GRU — GroupKFold-8 (by study group)")
res_cnngru_gkf = vgx.run_grouped_cv(
    make_cnngru, X, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=8),
    cv_groups=study_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler="plateau",
)
agg_cnngru_gkf = vg.aggregate(res_cnngru_gkf)
print(f"\n  MAE  {agg_cnngru_gkf['mae']:.4f} ± {agg_cnngru_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_cnngru_gkf['rmse']:.4f} ± {agg_cnngru_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_cnngru_gkf['r2']:.4f} ± {agg_cnngru_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_cnngru_gkf['skill']:.4f}")
print(f"  Spearman  {agg_cnngru_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_cnngru_gkf,
    "CNN-GRU \u2014 GroupKFold-8 per-fold scatter (SOH)",
    save_path=RESULTS / "03_cnngru_gkf_per_fold.png",
)
vgx.per_fold_loss_curves_ext(
    res_cnngru_gkf,
    "CNN-GRU \u2014 GroupKFold-8 train/val loss (L1)",
    save_path=RESULTS / "03_cnngru_gkf_loss_curves.png",
)

## GroupKFold-8 comparison

In [ ]:
ARM_ORDER_GKF = [
    ("RF",      res_rf_gkf,     agg_rf_gkf),
    ("GRU",     res_gru_gkf,    agg_gru_gkf),
    ("+CNNGRU", res_cnngru_gkf, agg_cnngru_gkf),
]
LOWER_BETTER = {"MAE", "RMSE"}

agg_str, agg_num = {}, {}
for arm, _, agg in ARM_ORDER_GKF:
    agg_str[arm] = {
        "MAE":      f"{agg['mae']:.4f} \u00b1 {agg['mae_std']:.4f}",
        "RMSE":     f"{agg['rmse']:.4f} \u00b1 {agg['rmse_std']:.4f}",
        "R\u00b2":       f"{agg['r2']:.4f} \u00b1 {agg['r2_std']:.4f}",
        "Skill":    f"{agg['skill']:.4f}",
        "Spearman": f"{agg['spearman']:.4f}",
    }
    agg_num[arm] = {
        "MAE":      agg["mae"],
        "RMSE":     agg["rmse"],
        "R\u00b2":       agg["r2"],
        "Skill":    agg["skill"],
        "Spearman": agg["spearman"],
    }

df_str = pd.DataFrame(agg_str).T
df_num = pd.DataFrame(agg_num).T

def _bold_better(df_display, df_values, lower_better):
    styled = df_display.copy()
    for col in df_display.columns:
        if col in lower_better:
            best_idx = df_values[col].idxmin()
        else:
            best_idx = df_values[col].idxmax()
        styled.loc[best_idx, col] = f"**{df_display.loc[best_idx, col]}**"
    return styled

df_display = _bold_better(df_str, df_num, LOWER_BETTER)
print("=== GroupKFold-8 aggregate results ===")
display(df_display)

df_str.to_csv(RESULTS / "03_gru_comparison_gkf_metrics.csv")
print(f"Saved: {RESULTS / '03_gru_comparison_gkf_metrics.csv'}")

fold_rows = []
for arm, res, _ in ARM_ORDER_GKF:
    for fi, fold in enumerate(res):
        m = fold["metrics"]
        fold_rows.append({
            "Model":      arm,
            "Fold":       fi + 1,
            "HeldGroup":  ",".join(fold.get("held_group") or []),
            "HeldBids":   ",".join(fold.get("held_bids") or []),
            "MAE":        m["mae"],
            "RMSE":       m["rmse"],
            "R\u00b2":         m["r2"],
            "Skill":      m["skill"],
            "Spearman":   m["spearman"],
            "best_ep":    fold.get("best_epoch"),
        })

df_folds = pd.DataFrame(fold_rows)
df_pivot = df_folds.pivot_table(
    index=["Fold", "HeldGroup", "HeldBids"],
    columns="Model",
    values=["MAE", "RMSE", "R\u00b2", "Skill", "Spearman", "best_ep"],
).round(4)

print("\n=== Per-fold breakdown ===")
display(df_pivot)
df_pivot.to_csv(RESULTS / "03_gru_per_fold_gkf_metrics.csv")
print(f"Saved: {RESULTS / '03_gru_per_fold_gkf_metrics.csv'}")

## Leave-One-Dataset-Out (NASA ↔ CALCE transfer)

Two folds only:
- **hold-NASA**: train on CALCE, test on NASA
- **hold-CALCE**: train on NASA, test on CALCE

This reveals how well the voltage-grid representation generalises across cell chemistries / cycler protocols.

In [ ]:
print("RF \u2014 LODO")
res_rf_lodo = vgx.run_rf_grouped_cv(
    feat, y, groups, cidx,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
)
agg_rf_lodo = vg.aggregate(res_rf_lodo)
print(f"  MAE {agg_rf_lodo['mae']:.4f}  R\u00b2 {agg_rf_lodo['r2']:.4f}")

In [ ]:
print("GRU \u2014 LODO")
res_gru_lodo = vgx.run_grouped_cv(
    make_gru, X, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler="plateau",
)
agg_gru_lodo = vg.aggregate(res_gru_lodo)
print(f"  MAE {agg_gru_lodo['mae']:.4f}  R\u00b2 {agg_gru_lodo['r2']:.4f}")

In [ ]:
print("CNN-GRU \u2014 LODO")
res_cnngru_lodo = vgx.run_grouped_cv(
    make_cnngru, X, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler="plateau",
)
agg_cnngru_lodo = vg.aggregate(res_cnngru_lodo)
print(f"  MAE {agg_cnngru_lodo['mae']:.4f}  R\u00b2 {agg_cnngru_lodo['r2']:.4f}")

In [ ]:
ARM_ORDER_LODO = [
    ("RF",      res_rf_lodo,     agg_rf_lodo),
    ("GRU",     res_gru_lodo,    agg_gru_lodo),
    ("+CNNGRU", res_cnngru_lodo, agg_cnngru_lodo),
]

lodo_rows = []
for arm, res, _ in ARM_ORDER_LODO:
    for fold in res:
        m = fold["metrics"]
        held_ds = ",".join(fold.get("held_group") or [])
        lodo_rows.append({
            "Model":        arm,
            "Held dataset": held_ds,
            "MAE":          round(m["mae"],  4),
            "RMSE":         round(m["rmse"], 4),
            "R\u00b2":           round(m["r2"],   4),
            "Skill":        round(m["skill"], 4),
            "Spearman":     round(m["spearman"], 4),
            "n":            m["n"],
        })

df_lodo = pd.DataFrame(lodo_rows)
print("=== Leave-One-Dataset-Out results ===")
display(df_lodo.pivot_table(index="Held dataset", columns="Model",
                             values=["MAE", "RMSE", "R\u00b2", "Skill"], sort=False).round(4))

df_lodo.to_csv(RESULTS / "03_gru_comparison_lodo_metrics.csv", index=False)
print(f"Saved: {RESULTS / '03_gru_comparison_lodo_metrics.csv'}")

for arm, res, _ in ARM_ORDER_LODO:
    vgx.per_fold_scatter_ext(
        res,
        f"{arm} \u2014 LODO per-fold scatter (SOH)",
        save_path=RESULTS / f"03_{arm.lower().replace('+', 'cnn')}_lodo_per_fold.png",
    )

## Conclusion

Fill in after running:

- **GRU vs LSTM (GroupKFold-8):** does GRU match, beat, or trail LSTM? GRU typically trains faster (fewer params per step) and can generalise better on smaller datasets by avoiding over-parameterisation of the gating mechanism.
- **CNN-GRU vs CNN-LSTM:** does the CNN front-end benefit GRU the same way it did (or didn't) for LSTM?
- **LODO transfer:** compare GRU LODO gap (GroupKFold-8 MAE − LODO MAE) to LSTM's gap from notebook 02. A smaller gap suggests GRU learns a more chemistry-agnostic representation.
- **Key slide:** GRU vs LSTM comparison table. If GRU matches LSTM with fewer params per hidden unit, it is the preferred model for deployment (lower inference cost on embedded systems).